# Check final classifier + keyword accuracy
- Check at the major category level
- Load in the labelled data, find it in the full data file and take categories from there

In [9]:
from discovery_child_development.utils import jsonl_utils
from discovery_child_development import PROJECT_DIR

import pandas as pd

In [6]:
taxonomy_labels = PROJECT_DIR / "outputs/labels/evals_data/taxonomy_labels_eval_annotated.jsonl"

In [17]:
data_df = (
    pd.read_csv(PROJECT_DIR / "outputs/data/tables/full_data_final.csv")
    .fillna({"major_category": ""})
    .assign(major_category=lambda df: df.major_category.apply(
                lambda x: [x.strip() for x in x.split(",")]
            ))
    .explode("major_category")
)

In [67]:
data_df_ = (
    pd.read_csv(PROJECT_DIR / "outputs/data/tables/full_data_final.csv")
    .fillna({"minor_category": ""})
    .assign(minor_category=lambda df: df.minor_category.apply(
                lambda x: [x.strip() for x in x.split(",")]
            ))
    .explode("minor_category")
)

In [68]:
sorted(data_df_.minor_category.unique())

['',
 'AI',
 'Child protection',
 'Cognitive development',
 'Communication and language',
 'Community',
 'Expressive arts and design',
 'Games',
 'Genetics',
 'Health',
 'Immersive tech',
 'Inclusion',
 'Income',
 'Inequalities',
 'Infancy',
 'Internet',
 'Labour market',
 'Literacy',
 'Mathematics',
 'Mental health',
 'Mobile',
 'Neuroscience',
 'Non-tech assessments',
 'Nutrition & weight',
 'Operations',
 'Oral health',
 'Parenting',
 'Personal social emotional',
 'Physical development',
 'Policy',
 'Prenatal',
 'Preschool',
 'RCTs',
 'Sleep',
 'Social services',
 'Special educational needs']

In [69]:
manual_labels_to_minor_categories = {
    'AR VR': 'Immersive tech',
    'Child protection': 'Child protection',
    'Cognitive development': 'Cognitive development',
    'Communication and language': 'Communication and language',
    'Community': 'Community',
    'Data science and AI': 'AI',
    'Family and home': 'Parenting',
    'Genetics': 'Genetics',
    'Health': 'Health',
    'Inclusion': 'Inclusion',
    'Income': 'Income',
    'Inequalities': 'Inequalities',
    'Internet': 'Internet',
    'Labour market': 'Labour market',
    'Literacy': 'Literacy',
    'Mathematics': 'Mathematics',
    'Mental health': 'Mental health',
    'Mobile': 'Mobile',
    'Neuroscience': 'Neuroscience',
    'Nutrition and weight': 'Nutrition & weight',
    'Operations': 'Operations',
    'Oral health': 'Oral health',
    'Personal social emotional': 'Personal social emotional',
    'Physical': 'Physical development',
    'Prenatal': 'Prenatal',
    'Preschool': 'Preschool',
    'Robotics': 'Immersive tech',
    'Sleep': 'Sleep',
    'Social media': 'Internet',
    'Social services': 'Social services',
    'Special needs': 'Special educational needs',
    'Wearables': 'Immersive tech',
}


In [56]:
manual_labels_to_major_categories = {
    'AR VR': 'Technology',
    'Child protection': 'Health',
    'Cognitive development': 'Development & learning',
    'Communication and language': 'Development & learning',
    'Community': 'Society',
    'Data science and AI': 'Technology',
    'Family and home': 'Parenting',
    'Genetics': 'Biosciences',
    'Health': 'Health',
    'Inclusion': 'Society',
    'Income': 'Society',
    'Inequalities': 'Society',
    'Internet': 'Technology',
    'Labour market': 'Society',
    'Literacy': 'Development & learning',
    'Mathematics': 'Development & learning',
    'Mental health': 'Health',
    'Mobile': 'Technology',
    'Neuroscience': 'Biosciences',
    'Nutrition and weight': 'Health',
    'Operations': 'Childcare & preschool',
    'Oral health': 'Health',
    'Personal social emotional': 'Development & learning',
    'Physical': 'Health',
    'Prenatal': 'Health',
    'Preschool': 'Childcare & preschool',
    'Robotics': 'Technology',
    'Sleep': 'Health',
    'Social media': 'Technology',
    'Social services': 'Society',
    'Special needs': 'Development & learning',
    'Wearables': 'Technology',
}


In [70]:
manual_labels_df = (
    pd.DataFrame(jsonl_utils.load_jsonl(taxonomy_labels))
    .query("label in @manual_labels_to_major_categories")
    .assign(major_category=lambda df: df.label.apply(lambda x: manual_labels_to_major_categories[x]))
    .assign(minor_category=lambda df: df.label.apply(lambda x: manual_labels_to_minor_categories[x]))
)

In [153]:
# manual_labels_df.groupby("label").size()

In [103]:
final_decision = []
for i, row in manual_labels_df.iterrows():
    final_categories = data_df_.query("_id == @row.id").minor_category.to_list()
    final_major_categories = data_df.query("_id == @row.id").major_category.to_list()
    if (len(final_categories) == 0) or (len(final_major_categories) == 0):
        final_decision.append("none")
    elif (row.minor_category in final_categories) or (row.major_category in final_major_categories):
        final_decision.append("accept")
    else:
        final_decision.append("reject")


In [104]:
manual_labels_df['final_decision'] = final_decision

In [105]:
manual_labels_df = (
    manual_labels_df
    .assign(agreement = lambda df: df.final_decision == df.answer)
)

In [40]:
# manual_labels_df.query("major_category == 'Technology' and agreement == False")

In [92]:
manual_labels_df.head(1)

,id,text,source,prediction,label,meta,_datetime,_input_hash,_task_hash,_view_id,answer,_annotator_id,_session_id,user_input,_timestamp,major_category,minor_category,final_decision,agreement
0,W4285741356,Mobile Augmented Reality applied as a learning...,openalex,AR VR,AR VR,{'url': 'https://openalex.org/W4285741356'},20240201180103,297503821,404094696,blocks,accept,taxonomy_data-karlis,taxonomy_data-karlis,NaN,NaN,Technology,Immersive tech,accept,True


In [121]:
import utils
topics_df = utils.load_topic_data()

2024-07-09 23:55:11,430 - botocore.credentials - INFO - Found credentials in environment variables.
2024-07-09 23:55:12,900 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [124]:
topics_df.head(1)

,topic,type,subtype,name
0,genetics,Biosciences,Genetics,Genetics


In [131]:
pd.DataFrame(
    manual_labels_df
    # .query("source == 'patents'")
    .query("final_decision != 'none'")
    .query("answer != 'ignore'")
    .groupby("minor_category").agreement.mean()
# )
).reset_index().merge(topics_df, left_on="minor_category", right_on="subtype").sort_values(["type", "minor_category"])

,minor_category,agreement,topic,type,subtype,name
5,Genetics,0.775000,genetics,Biosciences,Genetics,Genetics
20,Neuroscience,0.648649,neuroscience,Biosciences,Neuroscience,Neuroscience
22,Operations,0.644444,operations,Child care & preschool,Operations,Operations
28,Preschool,0.551020,preschool,Child care & preschool,Preschool,Preschool
2,Cognitive development,0.666667,cognitive,Development & learning,Cognitive development,Cognitive development
3,Communication and language,0.659091,communication,Development & learning,Communication and language,Communication and language
16,Literacy,0.723404,literacy,Development & learning,Literacy,Literacy
17,Mathematics,0.854167,mathematics,Development & learning,Mathematics,Mathematics
25,Personal social emotional,0.727273,emotional,Development & learning,Personal social emotional,Personal social emotional
31,Special educational needs,0.736842,send,Development & learning,Special educational needs,Special educational needs


In [120]:
(
    manual_labels_df
    .query("minor_category == 'Communication and language' and agreement == False")
    .query("final_decision != 'none'")
    .sort_values('answer')
)

,id,text,source,prediction,label,meta,_datetime,_input_hash,_task_hash,_view_id,answer,_annotator_id,_session_id,user_input,_timestamp,major_category,minor_category,final_decision,agreement
170,CN-116863763-A,An enlightenment learning device. The inventio...,patents,Communication and language,Communication and language,{'url': 'https://patents.google.com/patent/CN1...,20240201180103,256134403,792291210,blocks,accept,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,reject,False
171,KR-20220113100-A,Dialog doll for child using detachable smart p...,patents,Communication and language,Communication and language,{'url': 'https://patents.google.com/patent/KR2...,20240201180103,1535122423,2107761517,blocks,accept,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,reject,False
175,CN-113163961-A,A multifunctional fence for early childhood ed...,patents,Communication and language,Communication and language,{'url': 'https://patents.google.com/patent/CN1...,20240201180103,-885937015,-1779383135,blocks,accept,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,reject,False
205,W3126388494,Pengembangan Keterampilan Kolaborasi pada Anak...,openalex,Communication and language,Communication and language,{'url': 'https://openalex.org/W3126388494'},20240201180103,-265333923,-363556137,blocks,accept,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,reject,False
178,CN-218497664-U,An interactive system for preschool education ...,patents,Communication and language,Communication and language,{'url': 'https://patents.google.com/patent/CN2...,20240201180103,-17953229,1182448126,blocks,accept,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,reject,False
207,W3037153345,The functional development of a premature baby...,openalex,Communication and language,Communication and language,{'url': 'https://openalex.org/W3037153345'},20240201180103,-86412707,-1052426996,blocks,ignore,taxonomy_data-natalie,taxonomy_data-natalie,Not sure how to label this- the focus is defin...,NaN,Development & learning,Communication and language,accept,False
208,W4281572738,Gesture Development in Chinese-Speaking Presch...,openalex,Communication and language,Communication and language,{'url': 'https://openalex.org/W4281572738'},20240201180103,-2045976623,-1163892661,blocks,ignore,taxonomy_data-natalie,taxonomy_data-natalie,It can be labelled as both Autism and language...,NaN,Development & learning,Communication and language,accept,False
193,W3136571939,Digital Music Play in Early Childhood. The cha...,openalex,Communication and language,Communication and language,{'url': 'https://openalex.org/W3136571939'},20240201180103,661469316,569505800,blocks,reject,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,accept,False
179,KR-102570971-B1,Character Hangeul learning tool and method of ...,patents,Communication and language,Communication and language,{'url': 'https://patents.google.com/patent/KR1...,20240201180103,942487311,1251554922,blocks,reject,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,accept,False
196,W4283158062,Initial Evaluation of the Need for and Impact ...,openalex,Communication and language,Communication and language,{'url': 'https://openalex.org/W4283158062'},20240201180103,1595975682,1327274116,blocks,reject,taxonomy_data-natalie,taxonomy_data-natalie,NaN,NaN,Development & learning,Communication and language,accept,False


In [118]:
data_df.query("id == 'CN-218943743-U'")

,id,text,dataset,topic,major_category,minor_category,topic_code,country_code,url,amount,_id
18809,CN-218943743-U,A multifunctional sleeping pad suitable for ne...,patents,Infancy,General,Infancy,infancy,CN,https://patents.google.com/patent/CN218943743U,NaN,CN-218943743-U


In [80]:
df_ = manual_labels_df[['id', 'text', 'source', 'label', 'answer', 'final_decision', 'major_category', 'minor_category', 'agreement']]
df_.head(50)

,id,text,source,label,answer,final_decision,major_category,minor_category,agreement
0,W4285741356,Mobile Augmented Reality applied as a learning...,openalex,AR VR,accept,accept,Technology,Immersive tech,True
1,W4241367487,IPads in The Early Childhood Science Classroom...,openalex,AR VR,accept,reject,Technology,Immersive tech,False
2,KR-102520287-B1,Kids care platform service system using metave...,patents,AR VR,accept,reject,Technology,Immersive tech,False
3,W4293868984,Research on the Design and Effect of Early Chi...,openalex,AR VR,accept,accept,Technology,Immersive tech,True
4,W4320159718,Establishment of Childhood's Discipline Charac...,openalex,AR VR,reject,reject,Technology,Immersive tech,True
5,W3136197371,Research on Content Design of Media Facade and...,openalex,AR VR,accept,accept,Technology,Immersive tech,True
6,W4366994823,Mathematics Learning With Augmented Reality in...,openalex,AR VR,accept,accept,Technology,Immersive tech,True
7,W4388941595,Child development beyond the nutrition-specifi...,openalex,AR VR,reject,reject,Technology,Immersive tech,True
8,W2999874598,Effects of Augmented Reality Mobile Apps on Ea...,openalex,AR VR,accept,accept,Technology,Immersive tech,True
9,W4210500527,Application of Augmented Reality Technology in...,openalex,AR VR,accept,accept,Technology,Immersive tech,True


,id,text,dataset,topic,major_category,minor_category,topic_code,country_code,url,amount,_id


In [136]:
# generate random sampels
samples = []
for cat in ['Health', 'Development & learning', 'Child care & preschool', 'Parenting', 'Society']:
    samples.append(
        data_df.query("major_category == @cat").sample(50)
    )
samples = pd.concat(samples, ignore_index=True)

In [138]:
samples.to_csv("sample.csv", index=False)

In [141]:
# generate random sampels
samples = []
for cat in ['Mobile', 'Internet', 'AI', 'Immersive tech']:
    samples.append(
        data_df.query("minor_category == @cat").sample(50)
    )
samples = pd.concat(samples, ignore_index=True)

In [142]:
samples.to_csv("sample_2.csv", index=False)

In [133]:
data_df.major_category.unique()

array(['', 'Health', 'General', 'Technology', 'Biosciences',
       'Development & learning', 'Child care & preschool', 'Parenting',
       'Society'], dtype=object)

## Check number of relevance labels

In [143]:
relevance_labels = PROJECT_DIR / "outputs/labels/evals_data/relevance_labels_eval_annotated.jsonl"

In [146]:
pd.DataFrame(jsonl_utils.load_jsonl(relevance_labels)).prediction.value_counts()

Not-specified    50
Relevant         48
Not-relevant     48
Name: prediction, dtype: int64

In [147]:
relevance_labels_gpt = PROJECT_DIR / "outputs/labels/relevance/relevance_labels.jsonl"

In [150]:
pd.DataFrame(jsonl_utils.load_jsonl(relevance_labels_gpt)).prediction.value_counts().sum()

4796